<a href="https://colab.research.google.com/github/sagarshakya27/GenAI/blob/main/GenP6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
torch.manual_seed(42)

# 1. Define the input tokens and basic dimensions
tokens = ["Man", "eat", "cheese"]
seq_len = len(tokens)

# Let's define embedding dimension (d_model) and projection dimensions (d_k, d_v)
d_model = 16
d_k = 8
d_v = 8

# 2. Initialize toy embeddings for our tokens
# In a real model, these would come from an embedding layer or previous encoder layer
embeddings = torch.randn(seq_len, d_model)
print("Input Embeddings (Hidden States):")
for token, emb in zip(tokens, embeddings):
    print(f"{token}: {emb[:4].tolist()}... (dimension: {d_model})")

# 3. Define Query (Q), Key (K), and Value (V) projection matrices
W_q = torch.randn(d_model, d_k)
W_k = torch.randn(d_model, d_k)
W_v = torch.randn(d_model, d_v)

Q = torch.matmul(embeddings, W_q)
K = torch.matmul(embeddings, W_k)
V = torch.matmul(embeddings, W_v)

print("\nQuery (Q) Shape:", Q.shape)
print("Key (K) Shape:", K.shape)
print("Value (V) Shape:", V.shape)

# 4. Raw Attention Scores (Q * K^T)
attn_scores = torch.matmul(Q, K.transpose(0, 1))

# 5. Attention Distribution (Softmax)
attn_distribution = F.softmax(attn_scores, dim=-1)

print("Attention Scores:\n", attn_scores.numpy())
print("\nAttention Distribution (Softmax weights):\n", attn_distribution.numpy())

# 6. Calculate Final Attention Output Vectors
attention_output = torch.matmul(attn_distribution, V)

print("Output Attention Vectors Shape:", attention_output.shape)
for token, val in zip(tokens, attention_output):
    print(f"{token} attention vector: {val.tolist()}")

Input Embeddings (Hidden States):
Man: [1.9269152879714966, 1.4872840642929077, 0.9007171988487244, -2.1055209636688232]... (dimension: 16)
eat: [1.6423169374465942, -0.1595974713563919, -0.4973975419998169, 0.439589262008667]... (dimension: 16)
cheese: [-1.3846737146377563, -0.8712361454963684, -0.223365917801857, 1.7173614501953125]... (dimension: 16)

Query (Q) Shape: torch.Size([3, 8])
Key (K) Shape: torch.Size([3, 8])
Value (V) Shape: torch.Size([3, 8])
Attention Scores:
 [[-10.705313 -11.606056  62.89357 ]
 [-84.69232   26.995045   6.62294 ]
 [ 69.42341    5.606927 -28.202374]]

Attention Distribution (Softmax weights):
 [[1.0874531e-32 4.4179772e-33 1.0000000e+00]
 [0.0000000e+00 1.0000000e+00 1.4207163e-09]
 [1.0000000e+00 1.9268761e-28 3.9937006e-43]]
Output Attention Vectors Shape: torch.Size([3, 8])
Man attention vector: [-0.28153765201568604, 2.5002551078796387, -4.627270221710205, -0.3331648111343384, -2.7299487590789795, 0.5726194381713867, 1.9096590280532837, 3.625087022

In [2]:
# Explicitly show the math for step-by-step computation
for i, q_token in enumerate(tokens):
    print(f"\n--- Calculations for Query token: '{q_token}' ---")
    q_vec = Q[i]

    # 1. Compute raw dot products with all keys
    raw_scores = []
    scaled_scores = []

    for j, k_token in enumerate(tokens):
        k_vec = K[j]
        dot_prod = torch.dot(q_vec, k_vec).item()
        scaled_prod = dot_prod / (d_k ** 0.5)
        raw_scores.append(dot_prod)
        scaled_scores.append(scaled_prod)
        print(f"  Key '{k_token}': Raw Dot Product = {dot_prod:.4f}, Scaled ({d_k} ** 0.5): {scaled_prod:.4f}")

    # 2. Compute Softmax manually to show the distribution
    exp_scores = np.exp(scaled_scores)


--- Calculations for Query token: 'Man' ---
  Key 'Man': Raw Dot Product = -10.7053, Scaled (8 ** 0.5): -3.7849
  Key 'eat': Raw Dot Product = -11.6061, Scaled (8 ** 0.5): -4.1034
  Key 'cheese': Raw Dot Product = 62.8936, Scaled (8 ** 0.5): 22.2362

--- Calculations for Query token: 'eat' ---
  Key 'Man': Raw Dot Product = -84.6923, Scaled (8 ** 0.5): -29.9433
  Key 'eat': Raw Dot Product = 26.9950, Scaled (8 ** 0.5): 9.5442
  Key 'cheese': Raw Dot Product = 6.6229, Scaled (8 ** 0.5): 2.3416

--- Calculations for Query token: 'cheese' ---
  Key 'Man': Raw Dot Product = 69.4234, Scaled (8 ** 0.5): 24.5449
  Key 'eat': Raw Dot Product = 5.6069, Scaled (8 ** 0.5): 1.9823
  Key 'cheese': Raw Dot Product = -28.2024, Scaled (8 ** 0.5): -9.9710
